# 8. Temporal fuzzy sets: rules that depend on the time of day

Some patterns change with time. In the occupancy data (a room monitored by
temperature, humidity, light and CO2 sensors), light is a strong occupancy
signal by day and a weak one at night. **Temporal fuzzy sets** attach a
time-conditioned weight to every fuzzy set: the membership to "Light IS
High" is scaled by how often that label wins in each period of the day.

`TemporalFuzzyRulesClassifier` then fits one rule base per period. This
notebook reads the occupancy files shipped with the demos, so run it from
the `Demos` directory or the repository root.

In [1]:
import pathlib

import numpy as np
import pandas as pd

from ex_fuzzy import fuzzy_sets as fs, temporal, utils

data_dir = next(path for path in (pathlib.Path('occupancy_data'), pathlib.Path('Demos/occupancy_data')) if path.exists())
columns = ['date', 'Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio']
frames = [pd.read_csv(data_dir / name, index_col=0) for name in ('datatraining.txt', 'datatest.txt')]
X_total = pd.concat([frame[columns] for frame in frames])
y_total = np.concatenate([frame['Occupancy'].to_numpy() for frame in frames])
print(len(X_total), 'observations')
X_total.head()

10808 observations


,date,Temperature,Humidity,Light,CO2,HumidityRatio
1,2015-02-04 17:51:00,23.18,27.2720,426.0,721.25,0.004793
2,2015-02-04 17:51:59,23.15,27.2675,429.5,714.00,0.004783
3,2015-02-04 17:53:00,23.15,27.2450,426.0,713.50,0.004779
4,2015-02-04 17:54:00,23.15,27.2000,426.0,708.25,0.004772
5,2015-02-04 17:55:00,23.10,27.2000,426.0,704.50,0.004757


## Periods of the day

`temporal_cuts` marks each observation with the periods it falls into, from
the `date` column, and `temporal_assemble` splits train and test data so
that every period is represented in both.

In [2]:
periods = [['00:00:00', '10:00:00'], ['11:00:00', '19:00:00'], ['20:00:00', '23:00:00']]   # morning, day, evening
markers = utils.temporal_cuts(X_total, cutpoints=periods, time_resolution='hour')
time_moments = np.array([utils.assign_time(index, markers) for index in range(len(X_total))])
print('observations per period:', np.bincount(time_moments))

(X_train, X_test, y_train, y_test), (train_markers, test_markers) = utils.temporal_assemble(X_total, y_total, temporal_moments=markers)
train_moments = np.array([utils.assign_time(index, train_markers) for index in range(len(X_train))])
test_moments = np.array([utils.assign_time(index, test_markers) for index in range(len(X_test))])

observations per period: [5184 3712 1912]


## Temporal partitions

The base partitions are the usual quantile ones. `create_tempVariables`
wraps every fuzzy set in a `temporalFS` that carries, per period, the
relative frequency with which the set was the strongest label.

In [3]:
features = X_total.drop(columns='date')
base_partitions = utils.construct_partitions(features, fs.FUZZY_SETS.t1)
temporal_partitions = utils.create_tempVariables(features.to_numpy(), time_moments, base_partitions)

light = temporal_partitions[2]
pd.DataFrame({fuzzy_set.name: fuzzy_set.tmp_function for fuzzy_set in light.linguistic_variables},
             index=['morning', 'day', 'evening']).round(2)

,Low,Medium,High
morning,1.00,0.4,0.52
day,0.26,1.0,1.00
evening,0.50,0.0,0.00


The table reads: in the evening, "Light IS High" almost never wins, so its
membership is scaled down in that period.

## Fitting one rule base per period

In [4]:
classifier = temporal.TemporalFuzzyRulesClassifier(
    nRules=10, nAnts=3, linguistic_variables=temporal_partitions,
    fuzzy_type=fs.FUZZY_SETS.temporal, tolerance=0.001, n_class=2)
classifier.fit(X_train.drop(columns='date'), y_train, n_gen=20, pop_size=20, time_moments=train_moments)

predictions = classifier.predict(X_test.drop(columns='date'), test_moments)
print(f'test accuracy: {np.mean(predictions == y_test):.3f}')

/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A

/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A

/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A

/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A

/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A

/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A

/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A

/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/fuminides/GitHub/pyenvs/datasci/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A

test accuracy: 0.980


In [5]:
import warnings

with warnings.catch_warnings():
    # Some periods contain a single class, for which scikit-learn's MCC warns.
    warnings.simplefilter('ignore', UserWarning)
    temporal.eval_temporal_fuzzy_model(classifier, X_train.drop(columns='date'), y_train,
                                       X_test.drop(columns='date'), y_test,
                                       time_moments=train_moments, test_time_moments=test_moments,
                                       plot_rules=False, print_rules=True, plot_partitions=False)

ACCURACY
Train performance: 0.9849468305482668
Test performance: 0.9798149705634988
------------
MATTHEW CORRCOEF
Train performance: 0.9608534105149912
Test performance: 0.9484463028611213
------------
MOMENT 0
------------
MATTHEW CORRCOEF
Train performance: 0.9704105834763924
Test performance: 0.9608758706172816
------------
MOMENT 1
------------
MATTHEW CORRCOEF
Train performance: 0.9391275576570688
Test performance: 0.9182965600572524
------------
MOMENT 2
------------
MATTHEW CORRCOEF
Train performance: 0.0
Test performance: 0.0
------------
Rules for time step: 0
----------------
Consequent: 0
IF Humidity IS Low AND Light IS Low AND CO2 IS Low WITH DS 0.08729181354529544, ACC 1.0
IF Light IS Low AND HumidityRatio IS High WITH DS 0.054034899519558054, ACC 1.0

Consequent: 1
IF Humidity IS Low AND CO2 IS High WITH DS 0.003295528466341068, ACC 1.0
IF CO2 IS High AND HumidityRatio IS High WITH DS 0.00663197568061927, ACC 0.9902597402597403
IF Humidity IS Medium AND Light IS High AND 

Each period gets its own rule base, printed above, so a rule such as
*Light IS High -> occupied* can be present by day and absent at night.